<h2>Bronze work incremental</h2>
<h6>incremental bronze ingestion with rerun-safe watermark logic</h6>

### Step1__ imports and setup
<h6> this cell imports the pyspark and delta helpers used in the notebook,switches to the correct catalog,and makes sure the Bronze schema exists before start loading

In [0]:
from pyspark.sql import functions as f
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
%sql
use catalog novacart_adb

In [0]:
%sql
create schema if not exists novacart_adb.bronze;

# step 2 __ Bronze control table
### the table stores the **watermark** for each table
###### It helps the pipeline remember:
######-     the latest timestamp already processed
######-     the latest primary key processed at that timestamp
######-     how many rows were written in the latest run

######This is what makes Bronze load incremental and rerun-safe 


In [0]:
# %sql
# drop table if exists novacart_adb.bronze.ingestion_control;
# drop table if exists novacart_adb.bronze.orders_raw;
# drop table if exists novacart_adb.bronze.payments_raw;
# drop table if exists novacart_adb.bronze.products_raw;


In [0]:
spark.sql("""
          create table if not exists novacart_adb.bronze.ingestion_control(
              layer string,
              table_name  string,
              ts_col string,
              pk_col string,
              last_successful_ts timestamp,
              last_successful_pk bigint,
              last_run_id string,
              rows_written bigint,
              run_status string,
              updated_at timestamp
          )
          using delta
          """)

#step --3  source table configuration

In [0]:
tables_config = {
    "orders":{"pk_col":"order_id","ts_col":"updated_at"},
    "products":{"pk_col":"product_id","ts_col":"updated_at"},
    "payments":{"pk_col":"payment_id","ts_col":"processed_at"}
    }
bronze_run_id = str(uuid.uuid4())
print("current bronze run id: ", bronze_run_id)

# Step 4 _ Helper functions

In [0]:
def get_last_sucessful_watermarks(table_name:str):
    ctrl = (
        spark.table("novacart_adb.bronze.ingestion_control")
        .filter(
        (f.col("layer") == 'bronze') &
        (f.col("table_name") == table_name) &
        (f.col("run_status") == 'success')
        ) 
        .orderBy(f.col("updated_at").desc()).limit(1)
    )
    rows = ctrl.collect()
    if not rows:
        return None
        
    return rows[0]['last_successful_ts'],rows[0]['last_successful_pk']

In [0]:
def upsert_bronze_control(table_name,ts_col,pk_col,last_ts,last_pk,rows_written,run_id):
    control_df = spark.createDataFrame(
        [(
            'bronze',
            table_name,
            ts_col,
            pk_col,
            last_ts,
            int(last_pk) if last_pk is not None else None,
            run_id,
            int(rows_written),
            "success",
            datetime.now()
        )],
        schema = """
        layer string,
        table_name string,
        ts_col string,
        pk_col string,
        last_successful_ts timestamp,
        last_successful_pk bigint,
        last_run_id string,
        rows_written bigint,
        run_status string,
        updated_at timestamp
        """
    )
    dt = DeltaTable.forName(spark, "novacart_adb.bronze.ingestion_control")
    (dt.alias("t").merge(control_df.alias("s"),
        "t.table_name = s.table_name and t.layer = s.layer")
        .whenMatchedUpdate(set={
            "ts_col":"s.ts_col",
            "pk_col":"s.pk_col",
            "last_successful_ts":"s.last_successful_ts",
            "last_successful_pk":"s.last_successful_pk",
            "last_run_id":"s.last_run_id",
            "rows_written":"s.rows_written",
            "run_status":"s.run_status",
            "updated_at":"s.updated_at"
            })
        .whenNotMatchedInsertAll()
        .execute() 
    )

# step 5 __ Bronze Incremental load loop

In [0]:
for table_name,cfg in tables_config.items():
    ts_col = cfg['ts_col']
    pk_col = cfg['pk_col']
    source_table = f"`novacart_sql_connection_catalog`.dbo.{table_name}"
    target_table = f"`novacart_adb`.bronze.{table_name}_raw"
    last_successful_ts,last_successful_pk = get_last_sucessful_watermarks(table_name) or (None, None)

    if last_successful_ts is None:
        last_successful_ts = last_successful_ts.replace(
            microsecond=(last_successful_ts.microsecond//1000)*1000
        )
    print(f"\n === processing {table_name} ===")
    print(f"last successful ts: {last_successful_ts}")
    print(f"last successful pk: {last_successful_pk}")

    
    source_df = spark.read.table(source_table).withColumn(ts_col, f.date_trunc("MILLISECOND", f.col(ts_col).cast("timestamp")))

    if last_successful_ts is None:
        rows_to_load = source_df
    else:
        if last_successful_pk is None:
            rows_to_load = source_df.filter(f.col(ts_col) > f.lit(last_successful_ts))

        else:
            rows_to_load = source_df.filter(
                (f.col(ts_col) >f.lit(last_successful_ts)) |
                (
                    (f.col(ts_col) == f.lit(last_successful_ts))
                    & (f.col(pk_col).cast("long") > f.lit(last_successful_pk))
                )
            )
    rows_to_load = rows_to_load\
                        .withColumn("bronze_ingestion_at",f.current_timestamp())\
                        .withColumn("bronze_run_id",f.lit(bronze_run_id))\
                        .withColumn('bronze_source_table',f.lit(source_table))
    rows_count = rows_to_load.count()
    print(f"{table_name}rows to load: {rows_count}")

    if rows_count ==0:
        print(f"no new rows to load for {table_name}")
        upsert_bronze_control(table_name,
                              ts_col,
                              pk_col,
                              last_successful_ts,
                              last_successful_pk,
                              rows_count,
                              bronze_run_id)
        continue
    
    rows_to_load.write.format("delta").mode("append").saveAsTable(target_table)
    # max_ts = rows_to_load.agg(f.max(ts_col).alias('max_ts')).collect()[0]['max_ts']
    # max_pk = rows_to_load.filter(f.col(ts_col) == f.lit(max_ts))\
    #         .agg(f.max(f.col(pk_col).cast("long")).alias('max_pk'))\
    #         .collect()[0]['max_pk']

    watermarks_row = (
        rows_to_load
        .select(ts_col,pk_col)
        .orderBy(
            f.col(ts_col).desc(),
            f.col(pk_col).cast("long").desc()
        )
        .limit(1)
        .collect()[0]
    )
    max_ts = watermarks_row[ts_col]
    max_pk = watermarks_row[pk_col]


    upsert_bronze_control(table_name,
                        ts_col,
                        pk_col,
                        max_ts,
                        max_pk,
                        rows_count,
                        bronze_run_id)
    print(f"loaded {rows_count} rows for {table_name}")


## step 6 : Quick validation

In [0]:
print("orders bronze count:",spark.sql("select count(*) from `novacart_adb`.bronze.orders_raw").collect()[0][0])
print("products bronze count:",spark.sql("select count(*) from `novacart_adb`.bronze.products_raw").collect()[0][0])
print("payments bronze count:",spark.sql("select count(*) from `novacart_adb`.bronze.payments_raw").collect()[0][0])

display(spark.sql("select * from `novacart_adb`.bronze.ingestion_control").orderBy("table_name"))
